In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

train = pd.read_csv('trainingData.csv')
val_data = pd.read_csv('validationData.csv')
print(train.shape, val_data.shape)

(19937, 529) (1111, 529)


In [2]:
X_train = train.iloc[:, :520].replace(100, -105)
X_test = val_data.iloc[:, :520].replace(100, -105)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("预处理完成")

预处理完成


In [3]:
rf_bld = RandomForestClassifier(n_estimators=100, random_state=42)
rf_bld.fit(X_train, train['BUILDINGID'])
print("随机森林 建筑准确率:", accuracy_score(val_data['BUILDINGID'], rf_bld.predict(X_test)))

knn_bld = KNeighborsClassifier(n_neighbors=5)
knn_bld.fit(X_train, train['BUILDINGID'])
print("KNN 建筑准确率:", accuracy_score(val_data['BUILDINGID'], knn_bld.predict(X_test)))

随机森林 建筑准确率: 0.9990999099909991
KNN 建筑准确率: 0.9963996399639964


In [4]:
y_xy = train[['LONGITUDE', 'LATITUDE']].values
y_xy_test = val_data[['LONGITUDE', 'LATITUDE']].values

def report_error(name, pred):
    err = np.sqrt(((pred - y_xy_test) ** 2).sum(axis=1))
    print(f"{name}: 平均误差 {err.mean():.2f} 米, 中位数 {np.median(err):.2f} 米, 90%的点误差小于 {np.percentile(err, 90):.2f} 米")

knn_xy = KNeighborsRegressor(n_neighbors=5)
knn_xy.fit(X_train, y_xy)
report_error("KNN", knn_xy.predict(X_test))

rf_xy = RandomForestRegressor(n_estimators=100, random_state=42)
rf_xy.fit(X_train, y_xy)
report_error("随机森林", rf_xy.predict(X_test))

y_scaler = StandardScaler()
y_xy_s = y_scaler.fit_transform(y_xy)
mlp_xy = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
mlp_xy.fit(X_train_s, y_xy_s)
report_error("MLP", y_scaler.inverse_transform(mlp_xy.predict(X_test_s)))

KNN: 平均误差 8.95 米, 中位数 5.35 米, 90%的点误差小于 19.26 米
随机森林: 平均误差 12.51 米, 中位数 8.07 米, 90%的点误差小于 29.69 米
MLP: 平均误差 19.95 米, 中位数 12.30 米, 90%的点误差小于 38.97 米


In [5]:
preds = {
    "KNN": knn_xy.predict(X_test),
    "随机森林": rf_xy.predict(X_test),
    "MLP": y_scaler.inverse_transform(mlp_xy.predict(X_test_s)),
}

for name, pred in preds.items():
    err = np.sqrt(((pred - y_xy_test) ** 2).sum(axis=1))
    print(f"{name}: 5米内 {(err <= 5).mean():.1%}, 10米内 {(err <= 10).mean():.1%}, 20米内 {(err <= 20).mean():.1%}")

KNN: 5米内 47.0%, 10米内 73.7%, 20米内 90.6%
随机森林: 5米内 31.7%, 10米内 58.9%, 20米内 82.4%
MLP: 5米内 13.5%, 10米内 38.7%, 20米内 71.1%


In [6]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=1).fit(y_xy)
d, _ = nn.kneighbors(y_xy_test)
d = d.ravel()
print(f"验证点到最近训练采集点的距离：中位数 {np.median(d):.2f} 米，平均 {d.mean():.2f} 米")
print(f"距离超过 3 米的验证点占比: {(d > 3).mean():.1%}")

验证点到最近训练采集点的距离：中位数 0.80 米，平均 1.47 米
距离超过 3 米的验证点占比: 11.3%


In [7]:
from sklearn.model_selection import GroupShuffleSplit

train['location_group'] = train['LONGITUDE'].round(0).astype(str) + '_' + train['LATITUDE'].round(0).astype(str)
train['UNIQUE_ROOM'] = train['BUILDINGID'].astype(str) + '_' + train['FLOOR'].astype(str) + '_' + train['SPACEID'].astype(str)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx2, val_idx2 = next(gss2.split(X_train, train['UNIQUE_ROOM'], groups=train['location_group']))
y_tr_final = train['UNIQUE_ROOM'].iloc[train_idx2]
y_val_final = train['UNIQUE_ROOM'].iloc[val_idx2]

tr_rooms = set(y_tr_final)
va_rooms = set(y_val_final)
seen = y_val_final.isin(tr_rooms).values

print("验证集里的房间数:", len(va_rooms))
print("其中训练集没见过的房间数:", len(va_rooms - tr_rooms))
print("验证集记录中属于未见房间的比例:", 1 - seen.mean())
print(train.groupby('UNIQUE_ROOM')['location_group'].nunique().describe())

验证集里的房间数: 168
其中训练集没见过的房间数: 111
验证集记录中属于未见房间的比例: 0.6315932572050027
count    735.000000
mean       1.269388
std        0.496117
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max        3.000000
Name: location_group, dtype: float64


In [8]:
clf_final = RandomForestClassifier(n_estimators=100, random_state=42)
clf_final.fit(X_train.iloc[train_idx2], y_tr_final)
y_pred_final = clf_final.predict(X_train.iloc[val_idx2])

print("整体准确率:", accuracy_score(y_val_final, y_pred_final))
print("只看训练集见过的房间，准确率:", (y_pred_final[seen] == y_val_final.values[seen]).mean())

整体准确率: 0.055736813485589995
只看训练集见过的房间，准确率: 0.15129151291512916


In [9]:
err_knn = np.sqrt(((knn_xy.predict(X_test) - y_xy_test) ** 2).sum(axis=1))
far = d > 3
print(f"离训练点较远(>3米)的验证点：共 {far.sum()} 个，KNN 中位数误差 {np.median(err_knn[far]):.2f} 米")
print(f"离训练点较近(≤3米)的验证点：共 {(~far).sum()} 个，KNN 中位数误差 {np.median(err_knn[~far]):.2f} 米")

离训练点较远(>3米)的验证点：共 126 个，KNN 中位数误差 8.75 米
离训练点较近(≤3米)的验证点：共 985 个，KNN 中位数误差 4.94 米
